# 01 — Pelatihan Model FER2013 untuk SIP-Ekspresi BK

Notebook ini melatih model pengenalan ekspresi wajah berbasis CNN ringan, lalu mengekspor model agar dapat dipakai oleh aplikasi Streamlit `app.py`.

**Konteks aplikasi:** bimbingan dan konseling.  
**Batas penting:** model hanya alat bantu observasi awal, bukan alat diagnosis psikologis atau asesmen klinis.

Dataset yang digunakan: FER2013 dari Kaggle (`msambare/fer2013`) atau dataset FER2013 lain dengan struktur folder `train/` dan `test/`.

## 0. Instalasi pustaka

Jalankan cell ini bila notebook dijalankan pada lingkungan baru seperti Google Colab, Jupyter lokal, atau server kampus.

In [ ]:
# Jalankan jika pustaka belum tersedia.
# Di Google Colab, sebagian pustaka mungkin sudah terpasang.
%pip install -q tensorflow opencv-python-headless pillow numpy pandas scikit-learn matplotlib kaggle

## 1. Import pustaka dan konfigurasi

Konfigurasi dibuat sederhana agar mudah direplikasi. Model disimpan ke folder `models/` dan dataset diletakkan pada folder `data/`.

In [ ]:
from pathlib import Path
import os
import json
import shutil
import zipfile
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data" / "fer2013"
MODEL_DIR = BASE_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True, parents=True)

IMG_SIZE = (48, 48)
BATCH_SIZE = 64
EPOCHS = 40
VAL_SPLIT = 0.15

print("TensorFlow:", tf.__version__)
print("Working directory:", BASE_DIR)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 2. Unduh atau siapkan dataset FER2013

Notebook mendukung dua opsi.

1. **Kaggle API**: gunakan token `kaggle.json`, lalu jalankan cell unduhan.
2. **Manual**: unduh dataset dari Kaggle, ekstrak ke `data/fer2013`, dan pastikan ada folder `train/` dan `test/`.

Struktur yang diharapkan:

```text
data/fer2013/
├── train/
│   ├── angry/
│   ├── disgust/
│   ├── fear/
│   ├── happy/
│   ├── neutral/
│   ├── sad/
│   └── surprise/
└── test/
    ├── angry/
    ├── disgust/
    ├── fear/
    ├── happy/
    ├── neutral/
    ├── sad/
    └── surprise/
```

In [ ]:
# Opsi A: unduh otomatis melalui Kaggle API.
# Syarat:
# 1) Punya akun Kaggle.
# 2) Unduh kaggle.json dari Account -> Create New API Token.
# 3) Letakkan kaggle.json di ~/.kaggle/kaggle.json atau set KAGGLE_USERNAME dan KAGGLE_KEY.

DATASET_SLUG = "msambare/fer2013"

def download_kaggle_dataset(dataset_slug: str, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    marker = out_dir / ".download_done"
    if marker.exists():
        print("Dataset tampaknya sudah pernah diunduh:", out_dir)
        return

    try:
        cmd = ["kaggle", "datasets", "download", "-d", dataset_slug, "-p", str(out_dir), "--unzip"]
        print("Menjalankan:", " ".join(cmd))
        subprocess.run(cmd, check=True)
        marker.write_text("done", encoding="utf-8")
        print("Dataset selesai diunduh.")
    except Exception as e:
        print("Unduh Kaggle gagal.")
        print("Pesan error:", e)
        print("Silakan gunakan opsi manual: unduh dataset dari Kaggle, lalu ekstrak ke data/fer2013.")

# Hapus tanda komentar pada baris berikut bila Kaggle API sudah siap.
# download_kaggle_dataset(DATASET_SLUG, DATA_DIR)

In [ ]:
# Opsi B: jika dataset diunduh manual sebagai ZIP, letakkan di folder data/
# lalu isi nama file ZIP di bawah ini. Jika tidak diperlukan, lewati cell ini.

ZIP_PATH = BASE_DIR / "data" / "archive.zip"  # ubah bila nama file berbeda

if ZIP_PATH.exists() and not (DATA_DIR / "train").exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(DATA_DIR)
    print("ZIP diekstrak ke:", DATA_DIR)
else:
    print("Tidak ada ZIP manual yang diekstrak. Lanjutkan bila folder train/test sudah tersedia.")

In [ ]:
def find_fer_root(base: Path) -> Path:
    candidates = [base] + [p for p in base.rglob("*") if p.is_dir()]
    for p in candidates:
        if (p / "train").exists() and (p / "test").exists():
            return p
    raise FileNotFoundError(
        f"Tidak menemukan struktur train/test di {base}. "
        "Unduh dataset dahulu atau sesuaikan DATA_DIR."
    )

FER_ROOT = find_fer_root(DATA_DIR)
TRAIN_DIR = FER_ROOT / "train"
TEST_DIR = FER_ROOT / "test"

print("FER root:", FER_ROOT)
print("Train classes:", sorted([p.name for p in TRAIN_DIR.iterdir() if p.is_dir()]))
print("Test classes :", sorted([p.name for p in TEST_DIR.iterdir() if p.is_dir()]))

## 3. Membaca dataset

Data dibaca dengan `image_dataset_from_directory`. Folder `train` dibagi menjadi train dan validation menggunakan `validation_split`.

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="int",
    color_mode="grayscale",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    validation_split=VAL_SPLIT,
    subset="training",
    seed=SEED,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="int",
    color_mode="grayscale",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    validation_split=VAL_SPLIT,
    subset="validation",
    seed=SEED,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels="inferred",
    label_mode="int",
    color_mode="grayscale",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Class names:", class_names)
print("Jumlah kelas:", num_classes)

with open(MODEL_DIR / "labels.json", "w", encoding="utf-8") as f:
    json.dump({"class_names": class_names}, f, ensure_ascii=False, indent=2)

In [ ]:
# Optimasi pipeline input.
AUTOTUNE = tf.data.AUTOTUNE
train_ds_cached = train_ds.cache().shuffle(2048, seed=SEED).prefetch(AUTOTUNE)
val_ds_cached = val_ds.cache().prefetch(AUTOTUNE)
test_ds_cached = test_ds.cache().prefetch(AUTOTUNE)

## 4. Eksplorasi singkat distribusi kelas

Class imbalance cukup umum pada FER2013, sehingga notebook menghitung `class_weight` untuk membantu proses pelatihan.

In [ ]:
def count_images_per_class(folder: Path, class_names):
    counts = {}
    for cls in class_names:
        class_dir = folder / cls
        counts[cls] = len([p for p in class_dir.rglob("*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]])
    return counts

train_counts = count_images_per_class(TRAIN_DIR, class_names)
test_counts = count_images_per_class(TEST_DIR, class_names)
dist_df = pd.DataFrame({
    "class": class_names,
    "train_count": [train_counts[c] for c in class_names],
    "test_count": [test_counts[c] for c in class_names],
})
dist_df

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(dist_df["class"], dist_df["train_count"])
plt.title("Distribusi kelas pada data train")
plt.xlabel("Kelas")
plt.ylabel("Jumlah gambar")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Class weight dihitung dari jumlah file dalam folder train.
y_for_weight = []
for idx, cls in enumerate(class_names):
    y_for_weight += [idx] * train_counts[cls]
y_for_weight = np.array(y_for_weight)

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_classes),
    y=y_for_weight
)
class_weight = {i: float(w) for i, w in enumerate(weights)}
class_weight

## 5. Visualisasi contoh gambar

Cell ini hanya untuk memastikan dataset terbaca dengan benar.

In [ ]:
plt.figure(figsize=(10, 8))
for images, labels in train_ds.take(1):
    for i in range(min(16, images.shape[0])):
        ax = plt.subplot(4, 4, i + 1)
        plt.imshow(images[i].numpy().squeeze(), cmap="gray")
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
plt.tight_layout()
plt.show()

## 6. Membangun model CNN ringan

Model sengaja dibuat relatif ringan agar dapat dideploy sebagai aplikasi Streamlit. Untuk versi penelitian, arsitektur dapat ditingkatkan ke EfficientNet/MobileNet transfer learning, tetapi untuk HKI program komputer, yang paling penting adalah aplikasi berjalan stabil dan alur kerjanya jelas.

In [ ]:
def build_light_cnn(input_shape=(48, 48, 1), num_classes=7):
    inputs = keras.Input(shape=input_shape)

    x = layers.RandomFlip("horizontal", seed=SEED)(inputs)
    x = layers.RandomRotation(0.08, seed=SEED)(x)
    x = layers.RandomZoom(0.08, seed=SEED)(x)

    x = layers.Rescaling(1.0 / 255.0)(x)

    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.20)(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.30)(x)

    x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(256, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.40)(x)

    outputs = layers.Dense(num_classes, activation="softmax")(x)
    model = keras.Model(inputs, outputs, name="SIP_Ekspresi_BK_LightCNN")
    return model

model = build_light_cnn(num_classes=num_classes)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## 7. Pelatihan model

Model terbaik berdasarkan `val_accuracy` disimpan ke `models/best_fer_model.keras`.

In [ ]:
checkpoint_path = MODEL_DIR / "best_fer_model.keras"

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        mode="max",
        patience=8,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
]

history = model.fit(
    train_ds_cached,
    validation_data=val_ds_cached,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=callbacks,
)

## 8. Kurva pelatihan

Perhatikan apakah akurasi validasi stagnan atau terjadi overfitting. Untuk aplikasi awal, hasil model FER2013 biasanya belum sempurna karena ekspresi wajah manusia memang ambigu dan dataset relatif kecil.

In [ ]:
hist = pd.DataFrame(history.history)

plt.figure(figsize=(7, 4))
plt.plot(hist["accuracy"], label="train_accuracy")
plt.plot(hist["val_accuracy"], label="val_accuracy")
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(hist["loss"], label="train_loss")
plt.plot(hist["val_loss"], label="val_loss")
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Evaluasi pada data test

Bagian ini menghasilkan akurasi test, classification report, dan confusion matrix. Simpan hasilnya sebagai bukti pengujian untuk dokumentasi program.

In [ ]:
best_model = keras.models.load_model(checkpoint_path)

test_loss, test_acc = best_model.evaluate(test_ds_cached, verbose=1)
print(f"Test loss    : {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

y_true = []
y_pred = []

for x_batch, y_batch in test_ds_cached:
    probs = best_model.predict(x_batch, verbose=0)
    y_true.extend(y_batch.numpy().tolist())
    y_pred.extend(np.argmax(probs, axis=1).tolist())

report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
report_df = pd.DataFrame(report).T
report_df

In [ ]:
report_df.to_csv(MODEL_DIR / "classification_report.csv", index=True)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(range(num_classes), class_names, rotation=45)
plt.yticks(range(num_classes), class_names)
plt.colorbar()
for i in range(num_classes):
    for j in range(num_classes):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.tight_layout()
plt.show()

metrics_summary = {
    "test_loss": float(test_loss),
    "test_accuracy": float(test_acc),
    "class_names": class_names,
    "model_path": str(checkpoint_path),
}
with open(MODEL_DIR / "metrics_summary.json", "w", encoding="utf-8") as f:
    json.dump(metrics_summary, f, ensure_ascii=False, indent=2)

metrics_summary

## 10. Uji inferensi satu gambar

Gunakan satu gambar dari data test untuk memastikan bentuk input dan output sesuai dengan aplikasi Streamlit.

In [ ]:
LABEL_ID = {
    "angry": "Marah",
    "disgust": "Jijik",
    "fear": "Takut/Cemas",
    "happy": "Senang",
    "neutral": "Netral",
    "sad": "Sedih",
    "surprise": "Terkejut",
}

for images, labels in test_ds.take(1):
    sample = images[0:1]
    true_idx = int(labels[0].numpy())
    probs = best_model.predict(sample, verbose=0)[0]
    pred_idx = int(np.argmax(probs))
    print("True :", class_names[true_idx], "-", LABEL_ID.get(class_names[true_idx], class_names[true_idx]))
    print("Pred :", class_names[pred_idx], "-", LABEL_ID.get(class_names[pred_idx], class_names[pred_idx]))
    print("Conf :", float(probs[pred_idx]))

    plt.imshow(sample[0].numpy().squeeze(), cmap="gray")
    plt.title(f"Pred: {LABEL_ID.get(class_names[pred_idx], class_names[pred_idx])} ({probs[pred_idx]:.1%})")
    plt.axis("off")
    plt.show()
    break

## 11. Ekspor artefak untuk Streamlit

Pastikan file berikut tersedia:

```text
models/best_fer_model.keras
models/labels.json
models/classification_report.csv
models/metrics_summary.json
```

Setelah itu, jalankan aplikasi:

```bash
streamlit run app.py
```

In [ ]:
required_files = [
    MODEL_DIR / "best_fer_model.keras",
    MODEL_DIR / "labels.json",
    MODEL_DIR / "classification_report.csv",
    MODEL_DIR / "metrics_summary.json",
]

for f in required_files:
    print(f.name, "OK" if f.exists() else "BELUM ADA")

print("\nJika semua OK, jalankan:")
print("streamlit run app.py")

## 12. Catatan pengembangan lanjutan

Untuk versi yang lebih kuat, beberapa pengembangan yang dapat dilakukan:

1. Menggunakan transfer learning MobileNetV2/EfficientNet dengan input RGB.
2. Menambahkan deteksi wajah yang lebih kuat, misalnya MediaPipe Face Detection.
3. Menambahkan modul riwayat konseling berbasis persetujuan pengguna.
4. Menambahkan fitur analitik agregat per sesi, bukan per individu.
5. Melakukan uji fairness sederhana terhadap variasi pencahayaan, usia, jenis kamera, dan sudut wajah.
6. Menambahkan halaman manual penggunaan dan lembar persetujuan penggunaan data.